In [41]:
import os
import pickle
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [42]:
data = pd.read_csv("Dataset/jena_climate_2009_2016.csv")
print(data.shape)
data.head()

(420551, 15)


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [45]:
print(data.columns.tolist())

['Date Time', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)']


In [46]:
temperature = data["T (degC)"].values
print("Number of temperature observations:", len(temperature))
print("First 10 temperatures:")
print(temperature[:10])

Number of temperature observations: 420551
First 10 temperatures:
[-8.02 -8.41 -8.51 -8.31 -8.27 -8.05 -7.62 -7.62 -7.91 -8.43]


In [47]:
MAX_SAMPLES = 100000
temperature = temperature[:MAX_SAMPLES]

In [48]:
train_size = int(len(temperature) * 0.80)
train_data = temperature[:train_size]
test_data = temperature[train_size:]

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

Training samples: 80000
Testing samples: 20000


In [49]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data.reshape(-1, 1))
test_scaled = scaler.transform(test_data.reshape(-1, 1))
train_scaled.shape

(80000, 1)

In [50]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [51]:
WINDOW_SIZE = 30
def create_sequences(data, window_size):
    X = []
    y = []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

In [53]:
X_train, y_train = create_sequences(train_scaled, WINDOW_SIZE)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (79970, 30, 1)
y_train: (79970, 1)


In [54]:
test_with_context = np.concatenate([train_scaled[-WINDOW_SIZE:], test_scaled])
X_test, y_test = create_sequences(test_with_context, WINDOW_SIZE)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_test: (20000, 30, 1)
y_test: (20000, 1)


In [55]:
model = Sequential(
    [
        SimpleRNN(
            64, activation="tanh", input_shape=(WINDOW_SIZE, 1), return_sequences=False
        ),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(1),
    ]
)

model.summary()

/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,337 (24.75 KB)

 Trainable params: 6,337 (24.75 KB)

 Non-trainable params: 0 (0.00 B)

In [56]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

In [57]:
log_dir = "logs/fit"
os.makedirs(log_dir, exist_ok=True)
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [58]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=15,
    batch_size=256,
    callbacks=[early_stopping, tensorboard_callback],
    verbose=1,
)

Epoch 1/15


250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 38ms/step - loss: 0.0163 - mae: 0.0894 - val_loss: 2.5816e-04 - val_mae: 0.0136
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - loss: 0.0059 - mae: 0.0578 - val_loss: 5.7812e-04 - val_mae: 0.0225
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0042 - mae: 0.0483 - val_loss: 3.0557e-04 - val_mae: 0.0157
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - loss: 0.0031 - mae: 0.0407 - val_loss: 8.1181e-05 - val_mae: 0.0064
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0023 - mae: 0.0344 - val_loss: 7.5973e-05 - val_mae: 0.0061
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - loss: 0.0017 - mae: 0.0297 - val_loss: 7.5603e-05 - val_mae: 0.0066
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - loss: 0.0013 - mae: 0.0264 - val_loss: 9.6374e-05 - val_mae: 0.0075
Epoch 8/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - loss: 0.0011 - mae: 0.0236 - val_loss: 1.5315e-04 - val_mae: 0.0107
Epoch 9/15
250/250 ━━━━━━

In [59]:
y_pred_scaled = model.predict(X_test, verbose=1)

625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step


In [60]:
y_pred = scaler.inverse_transform(y_pred_scaled)
y_actual = scaler.inverse_transform(y_test)

print("First 10 predictions:")
print(y_pred[:10].flatten())

print("\nFirst 10 actual values:")
print(y_actual[:10].flatten())

First 10 predictions:
[31.374176 31.332067 31.727907 31.787544 31.80216  32.058544 32.28519
 32.339752 32.40672  32.441128]

First 10 actual values:
[31.45 31.7  31.75 31.58 31.89 32.07 31.9  31.93 32.11 32.27]


In [61]:
model.save("model.keras")

print("Model saved successfully.")

Model saved successfully.


In [62]:
def predict_next_temperature(previous_temperatures):
    values = np.array(previous_temperatures).reshape(-1, 1)
    scaled = scaler.transform(values)
    sequence = scaled[-WINDOW_SIZE:]
    sequence = sequence.reshape(1, WINDOW_SIZE, 1)
    prediction_scaled = model.predict(sequence, verbose=0)
    prediction = scaler.inverse_transform(prediction_scaled)
    return prediction[0][0]

In [63]:
last_30 = test_data[:30]
prediction = predict_next_temperature(last_30)
print(f"Predicted next temperature: " f"{prediction:.2f} °C")

Predicted next temperature: 33.10 °C


In [64]:
def forecast_future(initial_sequence, steps=10):
    sequence = list(initial_sequence)
    predictions = []
    for _ in range(steps):
        last_window = np.array(sequence[-WINDOW_SIZE:]).reshape(-1, 1)
        scaled_window = scaler.transform(last_window)
        X_input = scaled_window.reshape(1, WINDOW_SIZE, 1)
        next_scaled = model.predict(X_input, verbose=0)[0][0]
        next_temperature = scaler.inverse_transform([[next_scaled]])[0][0]
        predictions.append(next_temperature)
        sequence.append(next_temperature)
    return predictions

In [65]:
initial_sequence = list(test_data[:WINDOW_SIZE])
future_predictions = forecast_future(initial_sequence, steps=10)
for i, temperature_value in enumerate(future_predictions, start=1):
    print(f"Step {i}: " f"{temperature_value:.2f} °C")

Step 1: 33.10 °C
Step 2: 33.23 °C
Step 3: 33.45 °C
Step 4: 33.42 °C
Step 5: 33.57 °C
Step 6: 33.67 °C
Step 7: 33.98 °C
Step 8: 34.11 °C
Step 9: 34.28 °C
Step 10: 34.51 °C
